# S6 temperature-gradient heating metric (Fig 3b, Table 2)

Cycle-averaged temperature perturbation ⟨ΔT⟩(z) over the last forcing cycle and its vertical gradient in K/H₀, for the two central and the two edge forced columns of S6.

* Heavy part: `checks/regress_heating_slope.py` loads S6, writes `outputs/s1_s6_azimuthal/cache/temperature_last_cycle_S6.pkl` (all forced columns), and checks the result against the root code (bit-identical profiles, same slope). This notebook only reads that cache.
* Method: `dsmclib.heating`. The background is a constant T₀ = 270 K, so the column-0 background issue of the density growth does not apply. The fit cells are the first above 150 and 320 km (153.5–327.4 km, 17 cells), and H₀ = 39.97 km.
* New compared with the root code: an OLS standard error and a 95% Student-t interval (dof = 15) for each slope. The interval reflects the scatter about the line only.
* The centre pair is (554, 555), the symmetric pair used since commit 449ba24. The manuscript's earlier 4.3 K/H₀ does not reproduce with this pair (4.53) or with the old pair (555, 556), which gives 4.38.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dsmclib import heating, plotting
from dsmclib.paths import output_dir

plotting.use_style(font_scale=1.2)
FIG_DIR = output_dir("s1_s6_azimuthal", "figures")
TAB_DIR = output_dir("s1_s6_azimuthal", "tables")
P26_SLOPE, P26_ERR, T23_SLOPE = 5.60, 0.68, 9.4   # K/H0, as quoted in the manuscript

temp = heating.cached_temperature_last_cycle("S6")
temp["altitudes"][[0, -1]], len(temp["azimuthal_indices"])

(array([153.50692691, 327.35515743]), 88)

## Table 2: centre and edge slopes

In [2]:
groups = heating.group_heating_slopes(temp)
table2 = pd.DataFrame(groups).T[["columns", "n", "slope_K_H", "se_K_H", "ci_half_K_H", "r2"]]
table2["vs radial (%)"] = 100 * (table2["slope_K_H"].astype(float) / P26_SLOPE - 1)
table2.loc["edge vs centre (%)", "slope_K_H"] = 100 * (groups["edge"]["slope_K_H"] / groups["center"]["slope_K_H"] - 1)
table2.to_csv(TAB_DIR / "table2_heating_slopes_S6.csv")
table2

,columns,n,slope_K_H,se_K_H,ci_half_K_H,r2,vs radial (%)
center,"[554, 555]",17,4.534865,0.203645,0.434058,0.970639,-19.020274
edge,"[511, 598]",17,2.429513,0.175307,0.373659,0.927557,-56.615845
edge vs centre (%),NaN,NaN,-46.425905,NaN,NaN,NaN,NaN


## Column-to-column variability across the footprint

The single-column slopes scatter by more than the fit interval of the pair average, so the quoted ± is a lower bound on the total uncertainty.

In [3]:
cols = np.asarray(temp["azimuthal_indices"])
per_col = pd.DataFrame([{"column": c, **heating.heating_slope(temp["altitudes"], temp["temp_last_cycle"][c])} for c in cols])
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.errorbar(per_col["column"], per_col["slope_K_H"], yerr=per_col["ci_half_K_H"], fmt="o", ms=3, lw=0.8)
for name, color in (("center", "C2"), ("edge", "C3")):
    g = groups[name]
    ax.axhspan(g["slope_K_H"] - g["ci_half_K_H"], g["slope_K_H"] + g["ci_half_K_H"], color=color, alpha=0.2,
               label=f"{name} pair {g['columns']}: {g['slope_K_H']:.2f} ± {g['ci_half_K_H']:.2f}")
ax.axhline(P26_SLOPE, color="k", ls="--", label="1D radial (P26)")
ax.set(xlabel="S6 forced column", ylabel=r"$d\langle \Delta T\rangle/dz$ (K/$H_0$)")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "heating_slope_per_column_S6.png", dpi=200, bbox_inches="tight")
centre = per_col[(per_col["column"] >= cols[len(cols) // 2] - 4) & (per_col["column"] <= cols[len(cols) // 2] + 3)]
centre[["column", "slope_K_H", "ci_half_K_H"]].round(3)

,column,slope_K_H,ci_half_K_H
40,551,4.652,0.685
41,552,4.672,0.414
42,553,4.813,0.886
43,554,4.751,0.710
44,555,4.319,0.713
45,556,4.442,0.575
46,557,4.901,0.651
47,558,5.142,0.920
